# 02 — Skill Sweep

Sweep Tactics from 50 to 120 and visualise how mean damage scales.

**Key insight:** Warrior damage formula uses `(Anatomy + Tactics) / 2`,
NOT the weapon skill (Swordsmanship). Sweeping Swordsmanship will
affect hit chance but not the damage multiplier.

In [ ]:
import logging
from pathlib import Path

from omega.model.constants import SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_ANATOMY, SKILLID_WRESTLING
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, ParameterSweep, Scenario, Variable, WeaponSpec, run_sweep,
)
from omega.reporting.tables import summary_table, format_table_html
from omega.reporting.plots import damage_vs_parameter
from omega.logging import setup_logging

# Suppress noisy stub warnings — only show errors in notebook output
setup_logging(level=logging.ERROR)

# Works whether CWD is the project root or the notebooks/ directory
SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)

In [ ]:
sweep = ParameterSweep(
    scenario=Scenario(
        attacker=CombatantSpec(
            name="Warrior",
            skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
            str_=100, dex_=100, int_=25,
            class_levels={"IsWarrior": 5},
            weapon=WeaponSpec(name="Broadsword", damage="3d6+2"),
        ),
        defender=CombatantSpec(
            name="Target",
            is_npc=True,
            str_=50, dex_=50, int_=50,
            hp=500,
            skills={SKILLID_WRESTLING: 60},  # NPC combat skill — affects hit chance
            armor=ArmorSpec(ar=30),
        ),
        iterations=100,
        base_seed=1234,
    ),
    variables=(
        Variable.from_range("attacker", f"skills.{SKILLID_TACTICS}", start=50, stop=120, step=10),
    ),
)

result = run_sweep(sweep, shard=shard)
print(f"{len(result.cells)} cells completed in {result.total_time:.1f}s")

In [ ]:
# Damage vs Tactics curve with p5-p95 shading
damage_vs_parameter(result, f"attacker.skills.{SKILLID_TACTICS}", title="Damage vs Tactics")

In [ ]:
# Full summary table
from IPython.display import HTML

rows = summary_table(result, stats=["mean", "mean_on_hit", "median", "min", "max", "p5", "p95", "hit_rate"])
HTML(format_table_html(rows))

## Elemental Resistance Sweep (V1.5)

Sweep the defender's fire protection from 0 to 100 and see how elemental damage
drops off. The weapon has `ElementalDamage` set to `"FIRE:50 PHYSICAL:50"` — half
fire, half physical. The defender's `FireProtection` CProp controls resistance.

In [ ]:
from omega.reporting.plots import elemental_vs_parameter

fire_sweep = ParameterSweep(
    scenario=Scenario(
        attacker=CombatantSpec(
            name="Warrior",
            skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
            str_=100, dex_=100, int_=25,
            class_levels={"IsWarrior": 5},
            weapon=WeaponSpec(
                name="Fire Broadsword",
                damage="3d6+2",
                properties={"ElementalDamage": "FIRE:50 PHYSICAL:50"},
            ),
        ),
        defender=CombatantSpec(
            name="Protected Target",
            is_npc=True,
            str_=50, dex_=50, int_=50,
            hp=500,
            skills={SKILLID_WRESTLING: 60},
            armor=ArmorSpec(ar=30),
        ),
        iterations=100,
        base_seed=1234,
    ),
    variables=(
        Variable.from_range("defender", "properties.FireProtection", start=0, stop=100, step=10),
    ),
)

fire_result = run_sweep(fire_sweep, shard=shard)
print(f"{len(fire_result.cells)} cells completed in {fire_result.total_time:.1f}s")

In [ ]:
# Total damage vs fire protection — shows physical portion stays constant
damage_vs_parameter(
    fire_result,
    "defender.properties.FireProtection",
    title="Total Damage vs Fire Protection",
)

In [ ]:
# Per-element damage curves — fire drops while physical stays flat
elemental_vs_parameter(
    fire_result,
    "defender.properties.FireProtection",
    title="Per-Element Damage vs Fire Protection",
)

In [ ]:
# Elemental summary table with V1.5 stat columns
rows = summary_table(
    fire_result,
    stats=["mean", "median", "p5", "p95", "hit_rate", "elem_total_net", "elem_total_gross"],
)
HTML(format_table_html(rows))